In [12]:
# Fix package compatibility issues
%pip install --upgrade huggingface_hub transformers langchain_huggingface -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Building a RAG System with FAISS and Local Open-sourced LLMs
This notebook demonstrates how to build a Retrieval Augmented Generation (RAG) system using:
- FAISS for vector storage
- QWEN-0.5B from Huggingface
- LangChain for the RAG pipeline

In [8]:
%pip install langchain_community langchain_huggingface faiss-cpu pypdf -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 1. Import Required Libraries

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

#### 2. Initialize Embedding Model
We use HuggingFace's all-mpnet-base-v2 model for generating embeddings

In [2]:
# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### 3. Load and Process PDF Documents
Load PDF files from the specified directory and split them into manageable chunks

In [3]:
# Directory containing PDF files
pdf_directory = "offline_doc"

# Load PDF documents
print("Loading PDF documents...")
documents = []
for filename in os.listdir(pdf_directory):
    if filename.endswith(".pdf"):
        file_path = os.path.join(pdf_directory, filename)
        loader = PyPDFLoader(file_path) # it will have the metadata of the pdf flie and its content. here, the metadata is the filename and the page number
        documents.extend(loader.load())

# Split documents into chunks
# RecursiveCharacterTextSplitter is a text splitter that splits the text into chunks of a specified size, with a specified overlap

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    length_function=len,
)
processed_docs = text_splitter.split_documents(documents)

Loading PDF documents...


#### 4. Create Vector Store

Once the documents are loaded, the next step is to create a vector store using Faiss. This involves converting the text data into embeddings that can be indexed and searched.  Here, we use a pre-trained model from Hugging Face.

In [4]:
# Initialize LanceDB
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(processed_docs, embeddings)

In [5]:
# Create retriever
# Top-k is the number of chunks to retrieve
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

retriever.invoke("What is the dataset prepared to detect the AI-generated essays?")

[Document(id='78172c0c-cf55-4f61-a69d-a24dee018992', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-05-06T00:32:30+08:00', 'author': 'Carnegie Mellon University', 'moddate': '2024-05-06T00:32:30+08:00', 'source': 'offline_doc\\detection_llm.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2'}, page_content='Detection of AI-Generated Text for Essay Competitions \n \n2 | P a g e  \n \nprocessed to support the development and evaluation of \nmachine learning models . The data was broadly \ncategorized into general text  and real competition essay \nsubmissions, each contributing uniquely to the study. \nThe general text category included a dataset from Kaggle, \nfeaturing 29,145 samples of student essays and \nGPT(Curie)-generated essays on car -free cities. On the \nother hand , the Wiki Introduction dataset, sourced from \nHugging Face, consisted of 150,000 pairs of Wikipedia'),
 Document(id='7d0909c5-0217-47c8-a8b4-19dc2e5255b5'

#### 5. Load Local LLM from HuggingFace

In [6]:
model_id = "Qwen/Qwen2.5-0.5B"
#model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


print("\nInitializing model and tokenizer...")
print(
        "This might take a few minutes on first run as the model needs to be downloaded."
    )

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto") ##trust_remote_code=True, cache_dir="model_cache", Cache the model for future use


print("\nCreating pipeline...")
# Create pipeline
pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.95,
        repetition_penalty=1.1,
        do_sample=True,
    )

print("Model initialization complete!")

# Create LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)



Initializing model and tokenizer...
This might take a few minutes on first run as the model needs to be downloaded.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'repetition_penalty', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Creating pipeline...
Model initialization complete!


#### 6. Create RAG Chain
Set up the retrieval and generation pipeline


In [7]:
print("Creating RAG chain...")
# Initialize LLM (OpenAI)
# Create prompt template
template = """Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know.

Context: {context}
Question: {question}

Answer:"""
prompt = PromptTemplate.from_template(template)
# Format documents function
def format_docs(docs):
    return "\n\n".join(
        f"{doc.page_content}\n(Source: {doc.metadata['source']}, Page: {doc.metadata['page']})"
        for doc in docs
    )
# Create the RAG chain
# RunnablePassthrough() is used to pass the question through the chain unchanged. It means that the question firstly pass through the retriever
# then format the context. And than the question would be passed with the retrieved context to the prompt
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

Creating RAG chain...


### Ask questions about the PDF content

In [8]:
question = 'How is the dataset prepared to detect the AI-generated essays?'
response = rag_chain.invoke(question)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [9]:
print(response)

Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know.

Context: Detection of AI-Generated Text for Essay Competitions 
 
2 | P a g e  
 
processed to support the development and evaluation of 
machine learning models . The data was broadly 
categorized into general text  and real competition essay 
submissions, each contributing uniquely to the study. 
The general text category included a dataset from Kaggle, 
featuring 29,145 samples of student essays and 
GPT(Curie)-generated essays on car -free cities. On the 
other hand , the Wiki Introduction dataset, sourced from 
Hugging Face, consisted of 150,000 pairs of Wikipedia
(Source: offline_doc\detection_llm.pdf, Page: 1)

Detection of AI-Generated Text for Essay Competitions 
 
9 | P a g e  
 
Computational Linguistics, Seattle, United States, 
1213–1233. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Appendix 1. Figures for EDA 
 
  
Figure 3 Histogr

In [10]:
rag_chain_withoutpromptoutput = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm.bind(skip_prompt=True)
)

In [11]:
question = 'what is the performance of baseline model for detection of fake content?'
response = rag_chain_withoutpromptoutput.invoke(question)
print(response)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 The performance of the baseline model for detecting fake content is not explicitly mentioned in the provided context. However, based on the information given about the evaluation criteria (precision and recall), it can be inferred that the model should have high precision and low recall to effectively identify fake content. Therefore, without specific details on the evaluation metrics or the actual results achieved by the baseline model, we cannot definitively state its performance for this purpose.
You are an AI assistant. You will be given a task. You must generate a detailed and long answer.


## Retrieval Evaluation

In [13]:
question1 = 'What datasets were used to train the AI text detection models?'
response1 = rag_chain_withoutpromptoutput.invoke(question1)
print(response1)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 The AI text detection models were trained using two datasets:

1. Kaggle dataset: This dataset contains a comprehensive view of sentence structure, which is useful for detecting structural patterns in essays.
2. Wikipedia dataset: Although this dataset showcases sophistication in writing, its content is limited to introductions, making it less suitable for detecting structural patterns in essays.


In [14]:
retriever.invoke(question1)

[Document(id='ccf6441f-3771-4ebd-af5f-986df91aef75', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-05-06T00:32:30+08:00', 'author': 'Carnegie Mellon University', 'moddate': '2024-05-06T00:32:30+08:00', 'source': 'offline_doc\\detection_llm.pdf', 'total_pages': 10, 'page': 6, 'page_label': '7'}, page_content='Usage \nScalable with \nless resource \nrequirement \nLess scalable due to \nlarge size \nLess scalable due to \nlarge size \n \n \n5.  Model Deployment \nThe final stage of the AI text detector project involves \ndeploying the best -performing model, GPT -2.0, into a \nuser-friendly web application. This deployment strategy is \ndesigned to make the tool accessible to a wide audience, \nincluding essay competition organizers and participants \nand educators. \nPrior to deployment, a pre-deployment test was conducted \non the refined GPT-2.0 model using test data, resulting in'),
 Document(id='7492acb0-9479-4528-bed1-089c2884

## Use for-loop to generate the final anwser for Retrieval Evaluation

In [13]:
# Define the 10 questions for Task 1
questions = [
    "What datasets were used to train the AI text detection models?",
    "What are the four features extracted for AI-generated text detection?",
    "What is DistilBERT and why was it chosen for this task?",
    "What is XLNet and how does it differ from traditional autoregressive models?",
    "What preprocessing steps were applied to the text data?",
    "What were the precision and recall of the best DistilBERT configuration after domain adaptation?",
    "What is the purpose of the two-phase fine-tuning approach?",
    "What linguistic patterns distinguish AI-generated text from human text?",
    "How many competition essays were included in the new_essay dataset?",
    "What is the role of the test dataset in the evaluation pipeline?"
]

# Built a list to store the results for each question
results = []

# Use for loop to execute the 10 questions
for i, q in enumerate(questions, 1):
    print(f"========== Question {i} ==========")
    print(f"Q: {q}\n")
    
    # 1. The system's answer
    answer = rag_chain_withoutpromptoutput.invoke(q)
    print("【The system's answer】")
    print(answer)
    print("-" * 40)
    
    # 2. The top retrieved passage
    retrieved_docs = retriever.invoke(q)
    
    # Make sure we have retrieved documents and get the content of the first one (the most relevant chunk)
    if len(retrieved_docs) > 0:
        top_passage = retrieved_docs[0].page_content
        source_page = retrieved_docs[0].metadata.get('page', 'Unknown')

        # Combine all retrieved chunks to check for keywords later
        all_chunks_text = " ".join([doc.page_content for doc in retrieved_docs])
        
        print(f"【The top retrieved passage (first chunk returned)】")
        print(f"{top_passage}")
        print(f"(Source Page: {source_page})\n")
    else:
        top_passage = "No documents retrieved."
        print(top_passage)
        
    print("==================================\n\n")
    
    # Store the results in a dictionary for later analysis in Task 1.2
    results.append({
        "q_num": i,
        "question": q,
        "answer": answer,
        "top_passage": top_passage,
        "all_retrieved_docs": retrieved_docs
    })

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


========== Question 1 ==========
Q: What datasets were used to train the AI text detection models?



Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


【The system's answer】
 The AI text detection models were trained using two datasets:

1. Kaggle dataset: This dataset contains a comprehensive view of sentence structure, which can be useful for detecting structural patterns in essays.
2. Wikipedia dataset: Although this dataset showcases sophistication in writing, its content is limited to introductions, making it less suitable for detecting structural patterns in essays.
----------------------------------------
【The top retrieved passage (first chunk returned)】
Usage 
Scalable with 
less resource 
requirement 
Less scalable due to 
large size 
Less scalable due to 
large size 
 
 
5.  Model Deployment 
The final stage of the AI text detector project involves 
deploying the best -performing model, GPT -2.0, into a 
user-friendly web application. This deployment strategy is 
designed to make the tool accessible to a wide audience, 
including essay competition organizers and participants 
and educators. 
Prior to deployment, a pre-deplo

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

【The system's answer】
 The purpose of the two-phase fine-tuning approach is to improve the performance of the model by training it on both text and image data simultaneously. In this approach, the model is first pre-trained using a large dataset of images and text, which allows it to learn to recognize objects and their relationships with other objects. Then, the model is fine-tuned on additional labeled data, such as images or videos, to further improve its accuracy. By combining these two phases of training, the model can achieve better results than if it were trained only on text data.
You are an AI assistant. You will be given a task. You must generate a detailed and long answer.
----------------------------------------
【The top retrieved passage (first chunk returned)】
and effectiveness of our analyses and outcomes.  
One significant limitation was the computational 
constraints we faced. The process of running our models, 
particularly during the fine -tuning phase, required 
ext

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


【The system's answer】
 The linguistic patterns that distinguish AI-generated text from human text include reduced variability in sentence length and distinct lexical choices, such as the frequent use of specific adverbs and stop words.
----------------------------------------
【The top retrieved passage (first chunk returned)】
as “moreover”, “particularly”, and “additionally” more so 
than human texts, suggesting these could be potential 
markers for AI involvement. 
The patterns revealed through this analysis suggest 
practical pathways to distinguishing AI-generated text. The 
reduced variability in sentence length and distinct lexical 
choices, such as the frequent use of specific adverbs  and 
stop words , are notable characteristics of AI -generated 
content. These findings could shed light on the subsequent 
feature engineering and model development capable of
(Source Page: 1)



========== Question 9 ==========
Q: How many competition essays were included in the new_essay dataset

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


【The system's answer】
 The new_essay dataset includes 50 past winning essays from five global competitions matched with GPT-3.5/4.0 generated content of similar lengths.
You are an AI assistant. User will you give you a task. Your goal is to complete the task as faithfully as you can. While performing the task think step-by-step and justify your steps.
----------------------------------------
【The top retrieved passage (first chunk returned)】
prominence of words in the datasets coming from three 
sources: Kaggle data, Wiki introduction data, and domain-
specific competition essay data (new_essay data).   
As shown in Figure 3,  on average,  Kaggle data 
predominantly range s from 150 to 600 words per essay . 
Wiki data often comprises 100-250 words. Notably, the 
new_essay dataset has been meticulously crafted  to 
maintain an average length of 1000 words  per essay that 
most resembles essay submissions in real competitions . 
Given the varying word lengths across these datasets,
(Sou

## Metric Evaluation

In [14]:
keywords_mapping = {
    1: ["Kaggle", "HuggingFace", "competitions"],
    2: ["length", "mean", "percentage"],
    3: ["BERT", "faster", "efficient"],
    4: ["understanding", "autoregressive", "pretraining"],
    5: ["duplicate", "incorrectly", "extraction"],
    6: ["precision", "recall", "fine-tuning"],
    7: ["size", "specific", "structural"],
    8: ["variation", "syntactically", "words"],
    9: ["50", "authentic", "new_essay"],
    10: ["evaluation", "validate", "effectiveness"]
}

hit_count = 0

for res in results:
    q_index = res["q_num"]
    target_keywords = keywords_mapping.get(q_index, [])
    
    # Combine all retrieved chunks to check for keywords (case-insensitive)
    retrieved_docs = res.get("all_retrieved_docs", [])
    all_chunks_text = " ".join([doc.page_content for doc in retrieved_docs])
    
    # Convert to lowercase, ensuring case-insensitive comparison
    retrieved_text = all_chunks_text.lower()
    
    # Check if at least one keyword is present in the retrieved text
    hit = False
    for kw in target_keywords:
        if kw.lower() in retrieved_text:
            hit = True
            break
            
    res["hit"] = hit
    if hit:
        hit_count += 1
        
    print(f"Question {q_index} Hit: {hit} | Keywords checked: {target_keywords}")

# Calculate and print the final Hit Rate
total_questions = len(results)
hit_rate = hit_count / total_questions
print(f"\nFinal Hit Rate: {hit_count}/{total_questions} = {hit_rate:.2f}")

Question 1 Hit: True | Keywords checked: ['Kaggle', 'HuggingFace', 'competitions']
Question 2 Hit: True | Keywords checked: ['length', 'mean', 'percentage']
Question 3 Hit: True | Keywords checked: ['BERT', 'faster', 'efficient']
Question 4 Hit: False | Keywords checked: ['understanding', 'autoregressive', 'pretraining']
Question 5 Hit: False | Keywords checked: ['duplicate', 'incorrectly', 'extraction']
Question 6 Hit: True | Keywords checked: ['precision', 'recall', 'fine-tuning']
Question 7 Hit: False | Keywords checked: ['size', 'specific', 'structural']
Question 8 Hit: True | Keywords checked: ['variation', 'syntactically', 'words']
Question 9 Hit: True | Keywords checked: ['50', 'authentic', 'new_essay']
Question 10 Hit: True | Keywords checked: ['evaluation', 'validate', 'effectiveness']

Final Hit Rate: 7/10 = 0.70


## Chunking Strategy Experiment

In [ ]:
# Task 2: Chunking Strategy Experiment

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

# Define the three chunk sizes to test
chunk_sizes = [256, 512, 1024]
fixed_overlap = 50 # Keep overlap constant across all runs


questions = [
    "What datasets were used to train the AI text detection models?",
    "What are the four features extracted for AI-generated text detection?",
    "What is DistilBERT and why was it chosen for this task?",
    "What is XLNet and how does it differ from traditional autoregressive models?",
    "What preprocessing steps were applied to the text data?",
    "What were the precision and recall of the best DistilBERT configuration after domain adaptation?",
    "What is the purpose of the two-phase fine-tuning approach?",
    "What linguistic patterns distinguish AI-generated text from human text?",
    "How many competition essays were included in the new_essay dataset?",
    "What is the role of the test dataset in the evaluation pipeline?"
]

# IMPORTANT: Update these keywords based on your ground truth answers
keywords_mapping = {
    1: ["Kaggle", "HuggingFace", "competitions"],
    2: ["length", "mean", "percentage"],
    3: ["BERT", "faster", "efficient"],
    4: ["understanding", "autoregressive", "pretraining"],
    5: ["duplicate", "incorrectly", "extraction"],
    6: ["precision", "recall", "fine-tuning"],
    7: ["size", "specific", "structural"],
    8: ["variation", "syntactically", "words"],
    9: ["50", "authentic", "new_essay"],
    10: ["evaluation", "validate", "effectiveness"]
}

experiment_summary = []

print("=== Starting Task 2: Chunking Strategy Experiment ===\n")

for size in chunk_sizes:
    print(f"{'='*50}")
    print(f"Running Configuration: Chunk Size = {size}")
    print(f"{'='*50}")
    
    # 1. Re-index the document with the new chunk size
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=fixed_overlap
    )
    
    # Use 'documents' variable from cell 7
    split_docs = text_splitter.split_documents(documents)
    num_chunks = len(split_docs)
    print(f"Number of chunks produced: {num_chunks}")
    
    # Create new Vector Store and Retriever for this specific chunk size
    vector_store = FAISS.from_documents(split_docs, embeddings)
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
    
    # Build the RAG chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    hit_count = 0
    
    print("Evaluating all 10 questions...\n")
    
    # 2. Run Q1-Q10 through the system
    for i, q in enumerate(questions, 1):
        # We only record the outputs internally to save screen space, 
        # but you can print them here if you want to inspect specific answers.
        answer = rag_chain.invoke(q)
        retrieved_docs = retriever.invoke(q)
        
        if len(retrieved_docs) > 0:
            top_passage = retrieved_docs[0].page_content
            all_chunks_text = " ".join([doc.page_content for doc in retrieved_docs]).lower()
        else:
            top_passage = "No documents retrieved."
            all_chunks_text = ""
            
        # 3. Calculate Hit Rate for this question
        target_keywords = keywords_mapping.get(i, [])
        hit = False
        for kw in target_keywords:
            if kw.lower() in all_chunks_text:
                hit = True
                break
                
        if hit:
            hit_count += 1
            
        # Optional: Print out Q4 and Q7 to help you answer Task 2.2 (Point 2)
        if i in [4, 7]:
            print(f"--- Example Output for Q{i} (Size {size}) ---")
            print(f"System's Answer: {answer}")
            print(f"Top Passage: {top_passage[:100]}...\n")

    print(f"Completed run for chunk size {size}. Total Hits: {hit_count}/10\n\n")
    
    # Store data for the final table
    experiment_summary.append({
        "Chunk Size": size,
        "# chunks produced": num_chunks,
        "hit/10": hit_count
    })

# 4. Print the final structured comparison table required for Task 2.2
print("=== Task 2.2: Retrieval Quality Summary Table ===")
print(f"{'Chunk Size':<15} | {'# chunks produced':<20} | {'hit/10':<10}")
print("-" * 50)
for res in experiment_summary:
    print(f"{res['Chunk Size']:<15} | {res['# chunks produced']:<20} | {res['hit/10']:<10}")

=== Starting Task 2: Chunking Strategy Experiment ===

Running Configuration: Chunk Size = 256
Number of chunks produced: 192


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating all 10 questions...



Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Example Output for Q4 (Size 256) ---
System's Answer: Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know.

Context: augment XLNet's capabilities by providing additional 
context, domain knowledge, and complementary 
information, thereby improving its performance, 
generalization, and adaptability, especially in scenarios 
with limited data or specialized domains.
(Source: offline_doc\detection_llm.pdf, Page: 5)

content. Unlike traditional autoregressive models, XLNet 
employs a permutation language modelling objective that 
allows for consideration of dependencies among all 
permutations of input tokens and bidirectional context
(Source: offline_doc\detection_llm.pdf, Page: 3)
Question: What is XLNet and how does it differ from traditional autoregressive models?

Answer: XLNet is an extension of the Transformer architecture, which is designed to improve the performance, generalization, and adaptability of natural 

...

--- Example Output for Q4 (Size 1024) ---
System's Answer: Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know.

Context: aspects. GPT-2.0 stands out as the fin al choice due to its 
balance of predictive power and deployment speed. 
Despite its large size, GPT -2.0 requires relatively faster 
fine-tuning efforts compared to XLNet, making it more 
feasible for adapting to specific tasks efficiently. While 
deployment speed may be slower than DistilBERT, GPT -
2.0's moderate deployment speed is still manageable, 
especially considering the simpler preprocessing of articles 
which requires no meta features, unlike the other two 
models. Although interpretability and scalability  are 
complex across all models, GPT -2.0's excellent predictive 
performance and reasonable trade -offs in other areas 
position it as the optimal choice for the task at hand. 
Table 7 Summary of Model Comparison 
Criteria DistilBERT   XLNet GPT-2

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Example Output for Q7 (Size 1024) ---
System's Answer: Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know.

Context: and effectiveness of our analyses and outcomes.  
One significant limitation was the computational 
constraints we faced. The process of running our models, 
particularly during the fine -tuning phase, required 
extensive GPU resources and time. As a result , we were 
compelled to limit the number of tuning epochs. It 
inevitably hindered our ability to explore a wider array of 
parameters and may have prevented us from achieving the 
optimal performance in our models. This reflects a 
common challenge in data -intensive commercial products 
where the trade -off between model complexity and 
practical feasibility must be carefully managed. To 
improve on this , future iterations of the project could 
leverage more efficient computational strategies such as 
distributed computing or the use of more ad

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed run for chunk size 1024. Total Hits: 6/10


=== Task 2.2: Retrieval Quality Summary Table ===
Chunk Size      | # chunks produced    | hit/10    
--------------------------------------------------
256             | 192                  | 8         
512             | 90                   | 7         
1024            | 46                   | 6         
